# Detailed `qwanaflow` workflow

## Typical use of qwanaflow

For most users, obtaining a dataset of cell measurements using the `qwanaflow` command should be sufficient for their analysis needs. The options for this command can be displayed as such:

In [ ]:
# The exclamation mark indicates that this command is run by the shell and not by python
# Indeed, qwanaflow should be launched from the command line
!qwanaflow --help  

For example, the complete workflow may be launched from the command line using the following command:

In [ ]:
# Again, the exclamation mark here indicates that this command is passed to the shell
!qwanaflow --ncores 4 --vm-threshold 0.1 ../tests/data/test_image.png test_output

Sometimes, however, the user may want more control over the analytical workflow. If this is your case, or if you simply want to better understand what `qwanaflow` does under the hood, please read on. This detailed discussion will also allow you to better understand the parameters passed as arguments to `qwanaflow`.

## Importing the qwanamiz utilities and other required packages

Below, we load the `qwanamiz` that we will need for this example as well as some other packages that are needed for the analysis.

In [ ]:
import qwanamiz.qwanamiz as qmiz                 # main qwanamiz functions
import numpy as np                               # efficient array operations
import pandas as pd                              # manipulate data in tabular format with DataFrames
import matplotlib.pyplot as plt                  # plotting library
import skimage.io                                # image input/output from scikit-image
import skimage.measure                           # image measurements from scikit-image
import skimage.segmentation                      # image segmentation from scikit-image
from scipy.ndimage import distance_transform_edt # function that computes distance to nearest lumen pixel
import os                                        # operating-system related utilities
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 10) # Setting figure size to 10x10 everywhere in the document

## Test image

The image we will use as a test image for this example displays a few growth rings from a subfossil *Picea mariana* sample. The image is 3000x3000 pixels, which is smaller than the typical images generated in quantitative wood anatomy, but suitable for demonstration purposes.

We can read the image as a numpy array under the name `bw_image` (black & white image). Zeroes and ones correspond to cell wall and lumen pixels, respectively.

In [ ]:
bw_image_path = os.path.join("..", "tests", "data", "test_image.png")
bw_image = skimage.io.imread(bw_image_path)

plt.imshow(bw_image, cmap = 'gray')

A critical step of the analysis workflow is labelling the image, which is performed using functionality provided by scikit-image. This step labels all the lumens from individual cells and assigns a distinct integer to each.

In [ ]:
labeled_image = skimage.measure.label(bw_image)
plt.imshow(labeled_image)

Once the cells are labeled, we can compute some lumen properties using scikit-image's `regionprops_table` function. Many properties are available and described in the [official documentation](https://scikit-image.org/docs/0.25.x/api/skimage.measure.html#skimage.measure.regionprops_table), but only a few are necessary for downstream analyses.

Prior to computation, we need to set the scaling from pixels to micrometers so we get measurements in micrometers. Here, this scaling parameter is 0.55043 micrometers per pixel because the resolution of our image is 46146 dpi. In typical `qwanaflow` usage, this value would be provided as a command-line argument.

In [ ]:
# Setting the scaling from pixels to micrometers
pix_to_um = 0.55043

# Computing the properties of the cell lumens and returning a pandas DataFrame
cell_df = qmiz.measure_lumens(labeled_image, spacing = pix_to_um)
cell_df.info()

Once we have these measurements, we can further refine the set of identified cells. Because the image binarization step is often imperfect, it is likely that groups of two or more distinct cells will appear merged together and show up in the output as large cells with an irregular shape. We can identify incorrectly merged cells using those properties and use a watershed-based segmentation algorithm to properly redefine cell lumen boundaries.

In [ ]:
# Cells with area > 500 squared micrometers and solidity < 0.95 will be considered by the watershed algorithm
# This function updates the labeled_image and cell_df with redefined values
# watershed_result is an integer array that identifies the cells whose ID was potentially redefined
labeled_image, cell_df, watershed_result = qmiz.adjust_labels(labeled_image, cell_df, scale = pix_to_um,
                                                                     area_threshold = 500, solidity_threshold = 0.95)

# Zooming in on three cells that were split by the segmentation algorithm
plt.imshow(watershed_result[2130:2250,820:1000])

As we can see from the brighter yellow cells in the `labeled_image` array, several cells had their lumens redefined this way by the segmentation algorithm.

In [ ]:
# The cells that were considered by the watershed algorithm are a brighter yellow because their index
# is greater than those of the cells that were defined by the original labeling
plt.imshow(labeled_image)

By now, we already have several of the measurements that one would expect from a quantitative wood anatomy dataset. However, we have yet to measure cell wall thickness, which we will do later in this workflow. In preparation for measuring cell walls, we compute an array that maps each cell wall pixel (i.e. each pixel that is not a lumen pixel) to its distance to the nearest cell lumen.

In [ ]:
# distance_map: array containing the distance to the nearest lumen pixel
# nearest_label_coords: an array containing the ID (label) of the nearest lumen
distance_map, nearest_label_coords = distance_transform_edt(labeled_image == 0,
                                                            sampling = pix_to_um,
                                                            return_indices = True)

plt.imshow(distance_map)

Based on the distance map and the map of the cell ID nearest to each cell wall pixel, we can expand the cells beyond their lumen to also include their cell walls.

In [ ]:
# The distance parameters indicates that two cell lumens further away than this
# value will not be considered adjacent
expanded_labels = qmiz.expand_cells(labeled_image,
                                    distance_map,
                                    nearest_label_coords,
                                    distance = 10,
                                    spacing = pix_to_um)

plt.imshow(expanded_labels)


At this stage, we also want to measure the area of the whole cells using the same approach as we did earlier for lumens.

In [ ]:
# Measuring cell area on whole cells
cell_df = qmiz.measure_cells(cell_df, expanded_labels, spacing = pix_to_um)

Now that the array assigning each pixel to a cell is completed, we can build the regional adjacency graph (referred to as the RAG from now onwards) that stores information on which cells are adjacent to each other. This data structure will allow us to draw radial files and measure cell walls across tangential and radial ring boundaries, among other things. This is achieved using the function `get_adjacent_labels`, which returns a set of tuples representing pairs of adjacent cells.

More conveniently, for the rest of the analysis, we will be working with a DataFrame that summarizes the properties of these adjacencies. This is achieved using the function `adjacency_dataframe`, which also computes the angle between the centroids of the two cells (`angle` column), the location of the point that is exactly midway between those centroids (`center` column), as well as the distance between the two centroids (`length`). For convenience, this DataFrame is indexed using the same cell-pair tuples that are in the `adj_graph` set.

In [ ]:
# Creating a DataFrame from the set of adjacencies
# This DataFrame is indexed using the tuples in the adj_graph set
adjacency = qmiz.adjacency_dataframe(expanded_labels, cell_df)
adjacency.info()

Determining which cell adjacencies are along the radial direction (from pith to bark) and which are along the tangential direction (parallel to ring boundaries) is a significant challenge. In `qwanamiz`, we solved this problem by using the fact that radial adjacencies tend to cluster around the same angle due to the way that wood cells grow. Therefore, by finding the peak of the distribution of adjacency angles, we should be able to automatically determine the direction along which growth occurs (radial direction).

However, because of naturally occurring irregularities in growth, the radial direction may not always be exactly the same across a given image. To account for this, the directionality analysis is done separately on discrete sections of the image. By default, calling `qwanaflow` as a command-line utility will automatically determine the number of rows and columns into which the image should be divided for this analysis, but the user can also set these values using the `--dir-nrows` and `--dir-ncols` arguments. Here, we will divide the image into three rows and two columns, which corresponds to what `qwanaflow` would have automatically determined.

The directionality analysis assumes that the adjacency angles follow a von Mises distribution, which is analogous to the normal distribution but for values distributed around a circle, and therefore wrapped at 0/360 degrees. Therefore, finding the peak of the distribution for each image section can be reduced to fitting the observed distribution to von Mises distribution parameters. The command-line argument `--vm-threshold` to `qwanaflow` determines how fast convergence in the search for these parameters will be reached. Lower values will result in more precise values, but longer computation time. Here, we use a more permissive value (0.1) than the stringent default (0.001) to speed up computation.

In [ ]:
# This function call returns the updated adjacency DataFrame and the parameters of the von Mises distributions
# num_rows corresponds to the --dir-nrows command-line parameter
# num_cols corresponds to the --dir-ncols command-line parameter
# convergence_threshold corresponds to the vm-threshold command-line parameter
adjacency, vm_parameters = qmiz.directionality(adjacency,
                                               image_height = bw_image.shape[0], 
                                               image_width = bw_image.shape[1],
                                               spacing = pix_to_um,
                                               num_rows = 3,
                                               num_cols = 2,
                                               convergence_threshold = 0.1)

# Let us look at the updated adjacency DataFrame
adjacency.info()

As can be seen from the above output, `directionality` has added two columns `lower_bound` and `upper_bound` to the DataFrame of adjacencies. These correspond, respectively, to the smallest and largest angle (**in radians**) that are considered to belong to radial adjacencies.

For debugging purposes, `qwanaflow` will write an image to disk showing the observed distribution of adajcency angles and the von Mises distributions that were fit. This plot, which can be disabled using the `--disable-plots` command-line argument but is otherwise produced by default, can be produced with the function `plot_angles`. Here, we can see that the von Mises distribution (red line) fits the empirical distribution (blue lines) extremely well.

In [ ]:
angle_plot = qmiz.plot_angles(params = vm_parameters, num_rows = 3, num_cols = 2)

Using the lower and upper bounds from the directionality analysis, we can classify cell adjacencies based on the angle between them. Since the main objective of `qwanaflow` is to measure cell properties, we will actually be classifying cell walls. Cell walls are along the tangential direction if they form the boundary between cells that are adjacent along the radial direction, and vice versa. The function `classify_edges` will accordingly add a `wall_classification` column to the adjacency DataFrame and accepts a tolerance parameter (in degrees) around the lower and upper bounds. This allows for cells that are radially adjacent but whose angle departs from the local mean angle to still be considered as such. This parameter is controllable from the command-line using the argument `--angle-tolerance`.

In [ ]:
# This function updates the adjacency DataFrame
# tolerance corresponds to the --angle-tolerance command-line argument
adjacency = qmiz.classify_edges(adjacency, tolerance = 5)
adjacency['wall_classification'].value_counts()

We can now use the classified edges to assign the cells to radial files. In `qwanaflow`, this is a two-step process.

- First, radial files are initialized from cells that have no radially adjacent cell to their left. Then, the radial files are grown towards the right by iteratively adding adjacent cells in the radial direction. This process is done for each radial file until they cannot grow anymore because a cell does not have a radially adjacent neighbor.

- Second, radial files are joined using a more permissive threshold for considering two adjacent cells as radial. Internally to the function, cell adjacencies are reclassified using a more premissive threshold for cells to be considered radially adjacent. This threshold is controlled using the `--stitch-angle-tolerance` command-line parameter, which defaults to 20 degrees.

The function `assign_radial_files` updates both the cell-level DataFrame (here, `cell_df`) and the DataFrame of adjacencies (here, `adjacency`) with critical information for the rest of the workflow.

In [ ]:
# stitch_angle_tolerarance corresponds to the --stitch-angle-tolerance command-line parameter
cell_df, adjacency = qmiz.assign_radial_files(cell_df, adjacency, stitch_angle_tolerance = 20)

While the adjacency-level DataFrame is simply updated to reflect changes in wall classification at the radial file joining step, the cell-level DataFrame now contains the following additional columns:

- `classification`: whether the cell is *isolated* (not part of a radial file), *extremity* (located at the beginning or end of a radial file), or *regular* (located somewhere in the middle a of radial file)
- `radial_file`: an integer identifying which radial file the cell belongs to
- `file_rank`: an integer identifying the position of the cell within its radial file
- `left_neighbor`: the identifier of the cell that precedes this one in the radial file, for quick look-ups
- `left_angle`: the angle (in degrees) of this cell relative to its left neighbor
- `right_neighbor` and `right_angle`: analogous to their left-neighbor counterparts

In [ ]:
cell_df.info()

Having the information on radial files and left/right-neighbors allows for more precise measurement of lumen diameter because the angle of radial and tangential directions can now be estimated more accurately from the local adjacencies. This is what the function `measure_diameters` does.

In [ ]:
cell_df = qmiz.measure_diameters(cell_df, spacing = pix_to_um)
cell_df.info()

Measuring the diameters added the following columns to the cell-level DataFrame:

- `diameter_rad`: the diameter of the lumen along the radial direction
- `diameter_tan`: the diameter of the lumen along the tangential direction
- `extr_rad`: the coordinates of the points along which the radial diameter was measured, as a pair of length-two tuples with (y,x) coordinates
- `extr_tan`: similar to `extr_rad`, but for the tangential diameter
- `mean_angle`: the average angle between this cell and its left and right neighbors; it is a local estimate of the radial direction

The final step of the workflow measures the thickness of the radial walls, which is done in two steps. First, the adjacencies along which the walls will be measured are determined using the `get_radial_walls` function, which uses the information on radial files and the local angle of the radial direction. Second, the cell wall thickness is actually measured along those adjacencies using the function `measure_wallthickness`. Since this operation is computationally expensive but easily parallelized on different measurements, this command allows for multiprocessing using the `--ncores` command-line argument.

In [ ]:
# Computing cell wall thickness
# The ncores parameter is the --ncores command-line argument
cell_df, adjacency = qmiz.measure_walls(cell_df, adjacency, distance_map,
                                        auto_pixelwidth = True,
                                        scale = pix_to_um,
                                        scan_width = 75,
                                        nprocesses = 4)

# Finally, the tangential cell wall thickness is averaged between the left and right cell walls to provide an overall value
cell_df["WallThickness"] = cell_df[["left_wall_thickness", "right_wall_thickness"]].mean(axis = 1, skipna = True)

The rest of the `qwanaflow` command writes the results of the analysis to disk.